# Training the Scout wake words

Builds three wake word classifiers for the ii-ren shell — **"hey scout"**,
**"okay scout"** and **"scout"** — in one pass.

Training all three together is close to free: the expensive downloads (a 17GB
negative feature set, impulse responses, a noise corpus) are shared, and only
sample generation and the final fit happen per phrase.

### Why this builds its own Python

Two of the dependencies — `piper-phonemize` (turns the phrase into phonemes so
it can be spoken) and `speexdsp-ns` (a hard dependency of `openwakeword`) —
publish wheels only up to **cp312**, and Colab now runs Python 3.13. On 3.13
`pip install openwakeword` fails outright, and every training run then dies with
`ModuleNotFoundError: No module named 'openwakeword'`.

So everything below runs in a **Python 3.11 environment** built with `uv`, where
both have wheels. Colab's own kernel is left alone and is only used to drive it.
This is also why the run is pinned less tightly than upstream's notebook: the
Python version was doing the damage, not the package versions.

### Before you run anything

**Runtime → Change runtime type → T4 GPU.** The setup cell stops if there isn't
one, because sample generation on a CPU turns 90 minutes into an overnight job.

Then: Runtime → Run all.


## 1. A GPU, and a Python that works


In [ ]:
import subprocess, sys

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')
print('GPU:', gpu.stdout.strip())
print('Colab kernel:', sys.version.split()[0], '(only used to drive the 3.11 env below)')


In [ ]:
# uv builds the 3.11 environment. Much faster than conda, and it fetches a
# standalone interpreter rather than fighting the system one.
!pip install -q uv
!uv python install 3.11
!uv venv --python 3.11 /content/oww --quiet

PY = '/content/oww/bin/python'
!{PY} -V

import os
# Colab exports MPLBACKEND=module://matplotlib_inline.backend_inline, which only
# works inside its own kernel - the shim is not installed in this venv. Anything
# importing matplotlib here (torchmetrics does, transitively) would die on it:
#   ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not
#   a valid value for backend
# Agg is the right answer for a headless run anyway. Set on the kernel so every
# `!` command and subprocess below inherits it.
os.environ['MPLBACKEND'] = 'Agg'
print('MPLBACKEND ->', os.environ['MPLBACKEND'])


## 2. Sources and dependencies

Everything lands in the 3.11 environment. `tensorflow` and `onnx_tf` are
deliberately absent: they are needed only for tflite export, which the shell has
no use for, and they are the most fragile pins in upstream's notebook.
`train.py` imports them inside its tflite path only, so skipping them is safe as
long as `--convert_to_tflite` is never passed. It isn't.


In [ ]:
# Guarded so the cell is safe to re-run: a runtime that already has these from
# an earlier attempt would otherwise fail with 'destination path already exists'.
![ -d openwakeword ] || git clone -q https://github.com/dscripka/openwakeword
![ -d piper-sample-generator ] || git clone -q https://github.com/rhasspy/piper-sample-generator
!ls -d openwakeword piper-sample-generator


In [ ]:
# Torch first, so the heavy one resolves on its own.
!uv pip install --python {PY} -q torch torchaudio

# The two that forced Python 3.11. If either fails, the environment is not 3.11.
!uv pip install --python {PY} -q piper-phonemize speexdsp-ns

# No `datasets` and no `torchcodec`: openwakeword itself never imports either
# (checked), they were only upstream's way of fetching data, and both are a
# moving target. prep_data.py uses plain HTTP instead.
!uv pip install --python {PY} -q \
    webrtcvad mutagen==1.47.0 torchinfo torchmetrics \
    speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
    acoustics pronouncing deep-phonemizer \
    scipy numpy tqdm pyyaml onnxruntime soundfile

# openwakeword itself. With 3.11 its speexdsp-ns dependency now resolves.
!uv pip install --python {PY} -q ./openwakeword

!{PY} -c "import openwakeword, torch, piper_phonemize; print('openwakeword OK; CUDA:', torch.cuda.is_available())"


If that last line did not print `openwakeword OK` with `CUDA: True`, stop here —
everything below depends on it, and the failure will otherwise resurface as nine
identical tracebacks at the training step.


In [ ]:
%%bash
# The multi-speaker LibriTTS model that speaks the phrases (204MB).
# One %%bash cell, not several `!` lines: a shell variable does not survive
# between them, and a bare assignment line is parsed as Python and kills the cell.
mkdir -p piper-sample-generator/models
M=piper-sample-generator/models/en_US-libritts_r-medium.pt
if [ -s "$M" ]; then
  echo "already present"
else
  wget -q --show-progress -c -O "$M" \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
fi
ls -lh piper-sample-generator/models/
# Without this the TTS step fails with no useful message.
test -s "$M" || { echo 'FAILED: the voice model did not download'; exit 1; }


## 3. Negative and augmentation data

The big one is the ACAV100M feature set at **17.3GB**. It is what teaches the
model everything that *isn't* the phrase, and it is the reason these models hold
up outside a quiet room. It looks stuck; it isn't.


In [ ]:
%%writefile prep_data.py
"""Room impulse responses and background noise, as 16kHz mono 16-bit wav.

Deliberately does NOT use the `datasets` library. Upstream's two sources have
both rotted, and the library changed underneath them as well:

  - AudioSet's .tar files became parquet, so the hardcoded tar URL now 404s and
    `tar` reports "This does not look like a tar archive" on the error page;
  - FMA is a loading script, and datasets 4+ removed script support entirely;
  - datasets 4/5 hand back an AudioDecoder rather than a dict with 'array', and
    need torchcodec installed to decode anything at all.

Plain files over HTTP plus ffmpeg has none of those failure modes. The impulse
responses are still the same MIT set upstream uses. The background noise is
ESC-50 - one zip, 2000 plain wavs, no library and no dataset server involved.
"""
import concurrent.futures as futures
import json
import os
import subprocess
import urllib.request
import zipfile

RIR_REPO = "davidscripka/MIT_environmental_impulse_responses"
RIR_TREE = f"https://huggingface.co/api/datasets/{RIR_REPO}/tree/main/16khz"
RIR_BASE = f"https://huggingface.co/datasets/{RIR_REPO}/resolve/main"
ESC50_ZIP = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"

os.makedirs("mit_rirs", exist_ok=True)
os.makedirs("background_clips", exist_ok=True)
os.makedirs("_raw", exist_ok=True)


def fetch(url, dest):
    request = urllib.request.Request(url, headers={"User-Agent": "scout-trainer"})
    with urllib.request.urlopen(request) as response, open(dest, "wb") as handle:
        while True:
            block = response.read(1 << 20)
            if not block:
                break
            handle.write(block)


def to_16k(source, dest):
    """The sources are 24-bit and 44.1kHz respectively; training wants neither."""
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", source,
                    "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", dest],
                   check=False)


def rirs():
    with urllib.request.urlopen(RIR_TREE) as response:
        entries = json.load(response)
    wavs = [e["path"] for e in entries if e["path"].endswith(".wav")]
    print(f"{len(wavs)} impulse responses")

    def one(path):
        name = os.path.basename(path)
        raw = f"_raw/rir_{name}"
        fetch(f"{RIR_BASE}/{path}", raw)
        to_16k(raw, f"mit_rirs/{name}")
        os.remove(raw)

    with futures.ThreadPoolExecutor(8) as pool:
        list(pool.map(one, wavs))


def backgrounds():
    archive = "_raw/esc50.zip"
    if not os.path.exists(archive):
        print("ESC-50 (~600MB)...")
        fetch(ESC50_ZIP, archive)
    with zipfile.ZipFile(archive) as zf:
        members = [m for m in zf.namelist() if m.endswith(".wav") and "/audio/" in m]
        print(f"{len(members)} background clips")
        zf.extractall("_raw/esc50", members=members)

    sources = []
    for root, _, files in os.walk("_raw/esc50"):
        sources += [os.path.join(root, f) for f in files if f.endswith(".wav")]

    def one(index_source):
        index, source = index_source
        to_16k(source, f"background_clips/esc50_{index}.wav")

    with futures.ThreadPoolExecutor(8) as pool:
        list(pool.map(one, enumerate(sources)))


rirs()
backgrounds()
print("RIRs:", len(os.listdir("mit_rirs")),
      " background:", len(os.listdir("background_clips")))



In [ ]:
# No AudioSet tar any more - that URL 404s and the download was an error page,
# which is what 'This does not look like a tar archive' actually meant.
!{PY} prep_data.py


In [ ]:
# Precomputed negative features (17.3GB) and the false-positive validation set.
# -c resumes, so a dropped connection costs minutes rather than the whole file.
B = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main'
!wget -q --show-progress -c -O validation_set_features.npy $B/validation_set_features.npy
!wget -q --show-progress -c -O openwakeword_features_ACAV100M_2000_hrs_16bit.npy \
    $B/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!ls -lh *.npy


## 4. The phrases

Mirrors `tools/wakeword/phrases.json` in the repo. Keep the two in step.

`custom_negative_phrases` is doing real work, not decoration: it teaches each
model what to *reject*. "scout" alone is one syllable and a common English word,
so it gets the longest list — it is the phrase most likely to misfire, and hard
negatives are the cheapest defence.


In [ ]:
PHRASES = [
    {
        'name': 'hey_scout',
        'phrase': 'hey scout',
        'negatives': ['scout', 'hey', 'hey scott', 'hey stout', 'a scout',
                      'boy scout', 'girl scout', 'scouting', 'hey sprout'],
    },
    {
        'name': 'okay_scout',
        'phrase': 'okay scout',
        'negatives': ['scout', 'okay', 'ok', 'okay scott', 'okay stout',
                      'boy scout', 'scouting', 'okay google'],
    },
    {
        'name': 'scout',
        'phrase': 'scout',
        # One syllable and a real word, so it needs the most help.
        'negatives': ['scott', 'stout', 'sprout', 'shout', 'spout', 'snout',
                      'scouts', 'scouting', 'boy scout', 'girl scout',
                      'scout out', 'scoot', 'about', 'ouch', 'south'],
    },
]

# The quality dial. Upstream's quick notebook uses 1000 and warns accuracy may
# be low; its thorough one uses tens of thousands and takes hours. This is a
# deliberate middle. Raise both together if a phrase still misfires after you
# have tuned its threshold against real recordings.
N_SAMPLES = 5000
N_SAMPLES_VAL = 1000
STEPS = 20000


In [ ]:
import yaml, copy

base = yaml.safe_load(open('openwakeword/examples/custom_model.yml'))

for phrase in PHRASES:
    config = copy.deepcopy(base)
    config['model_name'] = phrase['name']
    config['target_phrase'] = [phrase['phrase']]
    config['custom_negative_phrases'] = phrase['negatives']
    config['n_samples'] = N_SAMPLES
    config['n_samples_val'] = N_SAMPLES_VAL
    config['steps'] = STEPS
    config['output_dir'] = f"./models/{phrase['name']}"
    config['piper_sample_generator_path'] = './piper-sample-generator'
    config['rir_paths'] = ['./mit_rirs']
    config['background_paths'] = ['./background_clips']
    config['false_positive_validation_data_path'] = './validation_set_features.npy'
    config['feature_data_files'] = {
        'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}
    with open(f"{phrase['name']}.yaml", 'w') as handle:
        yaml.dump(config, handle)
    print('wrote', phrase['name'] + '.yaml')


## 5. Train

Three stages per phrase — speak the samples, augment them with the rooms and
noise fetched above, then fit — all through the 3.11 interpreter.

Each stage is checked. Upstream's notebook lets a failed stage fall through to
the next one, which is how a single broken install turned into nine identical
tracebacks and no models; this stops at the first real failure instead.


In [ ]:
import os, subprocess

TRAIN = 'openwakeword/openwakeword/train.py'
# Belt and braces alongside the kernel-level setting: this is the process that
# actually imports matplotlib, via torchmetrics.
ENV = {**os.environ, 'MPLBACKEND': 'Agg'}

for phrase in PHRASES:
    config = f"{phrase['name']}.yaml"
    print(f"\n{'=' * 60}\n  {phrase['phrase']}\n{'=' * 60}")
    for stage in ('--generate_clips', '--augment_clips', '--train_model'):
        print(f'\n--- {stage} ---', flush=True)
        # Captured and echoed rather than inherited: a subprocess writing to the
        # real stdout does not reliably reach the notebook, which is how a failing
        # stage once showed nothing at all before raising.
        result = subprocess.run([PY, TRAIN, '--training_config', config, stage],
                                capture_output=True, text=True, env=ENV)
        print(result.stdout[-4000:] if result.stdout else '(no stdout)')
        if result.returncode != 0:
            print('--- stderr ---')
            print(result.stderr[-4000:] if result.stderr else '(no stderr)')
            raise SystemExit(
                f"{phrase['name']} failed at {stage} (exit {result.returncode}). "
                'Fix this before the later phrases - they fail the same way.')
print('\nAll three trained.')


## 6. Check and collect

Each model is loaded and its signature checked before packaging. The shell reads
the window length off the model, but the feature width must be 96 — that is fixed
by the shared embedding model, and a mismatch would only surface as a shape error
on your desktop, an hour from now.


In [ ]:
%%writefile collect.py
import glob, os, shutil, sys
import onnxruntime as ort

names = sys.argv[1:]
os.makedirs('scout-wakewords', exist_ok=True)
found = []
for name in names:
    matches = glob.glob(f'models/{name}/*.onnx')
    if not matches:
        print(f'MISSING: {name} produced no .onnx')
        continue
    session = ort.InferenceSession(matches[0], providers=['CPUExecutionProvider'])
    shape = session.get_inputs()[0].shape
    assert shape[-1] == 96, f'{matches[0]}: feature width {shape[-1]}, expected 96'
    target = f'scout-wakewords/{name}.onnx'
    shutil.copy(matches[0], target)
    found.append(target)
    print(f'{name:<12} input={shape}  {os.path.getsize(target) / 1e6:.2f}MB')

assert found, 'Nothing trained successfully.'
shutil.make_archive('scout-wakewords', 'zip', 'scout-wakewords')
print('\nPacked', len(found), 'model(s).')


In [ ]:
!{PY} collect.py hey_scout okay_scout scout


In [ ]:
from google.colab import files
files.download('scout-wakewords.zip')


## Then, on your machine

```sh
unzip ~/Downloads/scout-wakewords.zip -d ~/.local/share/vynx-conduit/wakeword/
```

Settings → Services → Conduit → Wake word: pick a phrase, turn it on, and put
**sensitivity back to about the middle** — a very high sensitivity will fire at
the television.

**Then tune the threshold against real recordings.** The default is a starting
point, not a finding. `tools/wakeword/README.md` has the commands;
`wakeword.py --wav` scores a file so you can see where your voice lands and where
a podcast lands, and put the threshold between them.
